In [1]:
# ETL Pipeline in Python

In [2]:
!pip install pandas
import pandas as pd
import numpy as np
import pyodbc
from datetime import datetime

In [3]:
pip install xlrd

Note: you may need to restart the kernel to use updated packages.


In [4]:
# 1. LOAD & CLEAN SUPERSTORE
df = pd.read_excel('/Users/owner/Downloads/Sample - Superstore-2.xls', sheet_name='Orders')
print(df.head())

   Row ID        Order ID Order Date  Ship Date       Ship Mode Customer ID  \
0       1  CA-2016-152156 2016-11-08 2016-11-11    Second Class    CG-12520   
1       2  CA-2016-152156 2016-11-08 2016-11-11    Second Class    CG-12520   
2       3  CA-2016-138688 2016-06-12 2016-06-16    Second Class    DV-13045   
3       4  US-2015-108966 2015-10-11 2015-10-18  Standard Class    SO-20335   
4       5  US-2015-108966 2015-10-11 2015-10-18  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   

  Postal Code  Region       Product ID         Category Sub-Category  \
0       42420   South  FUR-BO-10

In [5]:
# Basic cleaning
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])
df = df.dropna(subset=['Customer ID', 'Order ID'])
print(df.head())

   Row ID        Order ID Order Date  Ship Date       Ship Mode Customer ID  \
0       1  CA-2016-152156 2016-11-08 2016-11-11    Second Class    CG-12520   
1       2  CA-2016-152156 2016-11-08 2016-11-11    Second Class    CG-12520   
2       3  CA-2016-138688 2016-06-12 2016-06-16    Second Class    DV-13045   
3       4  US-2015-108966 2015-10-11 2015-10-18  Standard Class    SO-20335   
4       5  US-2015-108966 2015-10-11 2015-10-18  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   

  Postal Code  Region       Product ID         Category Sub-Category  \
0       42420   South  FUR-BO-10

In [6]:
# 2. CREATE CONTACTS 
contacts = df[['Customer ID', 'Customer Name', 'Segment', 'Country', 
               'City', 'State', 'Postal Code', 'Region']].drop_duplicates().copy()
contacts.rename(columns={'Customer ID': 'ContactID', 'Customer Name': 'ContactName'}, inplace=True)
print(contacts.head())

   ContactID      ContactName    Segment        Country             City  \
0   CG-12520      Claire Gute   Consumer  United States        Henderson   
2   DV-13045  Darrin Van Huff  Corporate  United States      Los Angeles   
3   SO-20335   Sean O'Donnell   Consumer  United States  Fort Lauderdale   
5   BH-11710  Brosina Hoffman   Consumer  United States      Los Angeles   
12  AA-10480     Andrew Allen   Consumer  United States          Concord   

             State  Postal Code Region  
0         Kentucky        42420  South  
2       California        90036   West  
3          Florida        33311  South  
5       California        90032   West  
12  North Carolina        28027  South  


In [7]:
# Add ~25% synthetic prospect-only contacts (for non-converted leads)
num_extra = int(len(contacts) * 0.25)
extra_contacts = pd.DataFrame({
    'ContactID': [f'PROSPECT_{i}' for i in range(1, num_extra + 1)],
    'ContactName': [f'Prospect Lead {i}' for i in range(1, num_extra + 1)],
    'Segment': np.random.choice(['Consumer', 'Corporate', 'Home Office'], num_extra),
    'Country': 'United States',
    'City': 'Unknown',
    'State': 'Unknown',
    'Postal Code': '00000',
    'Region': np.random.choice(contacts['Region'].unique(), num_extra)
})
contacts = pd.concat([contacts, extra_contacts], ignore_index=True)
print(contacts.head())

  ContactID      ContactName    Segment        Country             City  \
0  CG-12520      Claire Gute   Consumer  United States        Henderson   
1  DV-13045  Darrin Van Huff  Corporate  United States      Los Angeles   
2  SO-20335   Sean O'Donnell   Consumer  United States  Fort Lauderdale   
3  BH-11710  Brosina Hoffman   Consumer  United States      Los Angeles   
4  AA-10480     Andrew Allen   Consumer  United States          Concord   

            State Postal Code Region  
0        Kentucky       42420  South  
1      California       90036   West  
2         Florida       33311  South  
3      California       90032   West  
4  North Carolina       28027  South  


In [8]:
# 3. CREATE DEALS (aggregated per Order ID)
deals = df.groupby('Order ID').agg({
    'Customer ID': 'first',
    'Order Date': 'first',
    'Sales': 'sum',
    'Quantity': 'sum',
    'Profit': 'sum',
    'Category': lambda x: ', '.join(x.unique())
}).reset_index()

deals.rename(columns={
    'Order ID': 'DealID',
    'Customer ID': 'ContactID',
    'Order Date': 'CloseDate',
    'Sales': 'DealAmount'
}, inplace=True)

deals['Stage'] = 'Closed Won'
deals['CloseDate'] = pd.to_datetime(deals['CloseDate'])
deals = deals[['DealID', 'ContactID', 'DealAmount', 'CloseDate', 'Stage', 'Category', 'Quantity', 'Profit']]
print(deals.head())

           DealID ContactID  DealAmount  CloseDate       Stage  \
0  CA-2014-100006  DK-13375     377.970 2014-09-07  Closed Won   
1  CA-2014-100090  EB-13705     699.192 2014-07-08  Closed Won   
2  CA-2014-100293  NF-18475      91.056 2014-03-14  Closed Won   
3  CA-2014-100328  JC-15340       3.928 2014-01-28  Closed Won   
4  CA-2014-100363  JM-15655      21.376 2014-04-08  Closed Won   

                     Category  Quantity    Profit  
0                  Technology         3  109.6113  
1  Furniture, Office Supplies         9  -19.0890  
2             Office Supplies         6   31.8696  
3             Office Supplies         1    1.3257  
4             Office Supplies         5    7.7192  


In [9]:
# 4. CREATE LEADS (real + synthetic non-converted) 
# Real leads (one per customer who has a deal)
first_deal_date = deals.groupby('ContactID')['CloseDate'].min()

leads_real = contacts[contacts['ContactID'].isin(first_deal_date.index)].copy()
leads_real['LeadID'] = leads_real['ContactID'] + '_L'
leads_real['LeadDate'] = leads_real['ContactID'].map(first_deal_date) - pd.to_timedelta(np.random.randint(5, 60, len(leads_real)), 'd')
leads_real['LeadSource'] = np.random.choice(['Website', 'Referral', 'Paid Ad', 'Cold Call', 'Email Campaign'], len(leads_real))
leads_real['LeadStatus'] = 'Converted'

# Synthetic non-converted leads (using the prospect-only contacts)
leads_lost = extra_contacts.copy()
leads_lost['LeadID'] = leads_lost['ContactID'] + '_L'
leads_lost['LeadDate'] = pd.to_datetime('2024-01-01') + pd.to_timedelta(np.random.randint(0, 200, len(leads_lost)), 'd')
leads_lost['LeadSource'] = np.random.choice(['Website', 'Referral', 'Paid Ad', 'Cold Call', 'Email Campaign'], len(leads_lost))
leads_lost['LeadStatus'] = 'Not Converted'

leads = pd.concat([leads_real, leads_lost], ignore_index=True)
print(leads.head())

  ContactID      ContactName    Segment        Country             City  \
0  CG-12520      Claire Gute   Consumer  United States        Henderson   
1  DV-13045  Darrin Van Huff  Corporate  United States      Los Angeles   
2  SO-20335   Sean O'Donnell   Consumer  United States  Fort Lauderdale   
3  BH-11710  Brosina Hoffman   Consumer  United States      Los Angeles   
4  AA-10480     Andrew Allen   Consumer  United States          Concord   

            State Postal Code Region      LeadID   LeadDate LeadSource  \
0        Kentucky       42420  South  CG-12520_L 2015-08-19   Referral   
1      California       90036   West  DV-13045_L 2016-05-06    Paid Ad   
2         Florida       33311  South  SO-20335_L 2015-08-20    Paid Ad   
3      California       90032   West  BH-11710_L 2014-04-27   Referral   
4  North Carolina       28027  South  AA-10480_L 2014-03-10  Cold Call   

  LeadStatus  
0  Converted  
1  Converted  
2  Converted  
3  Converted  
4  Converted  


In [10]:
# 5. CREATE ACTIVITIES 
np.random.seed(42)
num_activities = 800
activities = pd.DataFrame({
    'ActivityID': [f'ACT{str(i).zfill(5)}' for i in range(1, num_activities + 1)],
    'ContactID': np.random.choice(contacts['ContactID'].values, num_activities),
    'DealID': np.random.choice(deals['DealID'].values, num_activities),
    'ActivityType': np.random.choice(['Call', 'Email', 'Meeting', 'Demo', 'Task'], num_activities),
    'ActivityDate': pd.to_datetime('2023-01-01') + pd.to_timedelta(np.random.randint(-400, 400, num_activities), 'd'),
    'Notes': 'Synthetic activity for CRM demo'
})

In [11]:
import pyodbc

conn_str = (
    r'DRIVER={ODBC Driver 18 for SQL Server};'
    r'SERVER=localhost,1433;'
    r'UID=sa;'
    r'PWD=@Ayomikun123!;'
    r'TrustServerCertificate=yes;'
    r'Encrypt=no;'
)

try:
    # Connect without specifying a database
    conn = pyodbc.connect(conn_str, timeout=30)
    cursor = conn.cursor()
    
    # Turn off autocommit for this operation
    conn.autocommit = True
    
    # Create Database
    cursor.execute("""
        IF NOT EXISTS (SELECT * FROM sys.databases WHERE name = 'CRM_Analytics')
            CREATE DATABASE CRM_Analytics;
    """)
    
    print("✅ Database 'CRM_Analytics' created successfully (or already existed)!")
    
    cursor.close()
    conn.close()

except Exception as e:
    print("❌ Error:")
    print(e)

✅ Database 'CRM_Analytics' created successfully (or already existed)!


In [12]:
import pyodbc
import pandas as pd
import numpy as np

print("Starting ETL Process...\n")

# ====================== CONNECTION ======================
conn_str = (
    r'DRIVER={ODBC Driver 18 for SQL Server};'
    r'SERVER=localhost,1433;'
    r'DATABASE=CRM_Analytics;'
    r'UID=sa;'
    r'PWD=@Ayomikun123!;'
    r'TrustServerCertificate=yes;'
    r'Encrypt=no;'
)

conn = pyodbc.connect(conn_str, timeout=30)
cursor = conn.cursor()
print("✅ Connected to SQL Server successfully!\n")

Starting ETL Process...

✅ Connected to SQL Server successfully!



In [13]:
# ====================== CREATE TABLES ======================
table_schemas = {
    'Contacts': """
        CREATE TABLE Contacts (
            ContactID VARCHAR(20) PRIMARY KEY,
            ContactName NVARCHAR(255),
            Segment NVARCHAR(50),
            Country NVARCHAR(50),
            City NVARCHAR(100),
            State NVARCHAR(100),
            PostalCode NVARCHAR(20),
            Region NVARCHAR(50)
        )
    """,
    'Leads': """
        CREATE TABLE Leads (
            LeadID VARCHAR(30) PRIMARY KEY,
            ContactID VARCHAR(20),
            LeadDate DATETIME,
            LeadSource NVARCHAR(50),
            LeadStatus NVARCHAR(20)
        )
    """,
    'Deals': """
        CREATE TABLE Deals (
            DealID VARCHAR(20) PRIMARY KEY,
            ContactID VARCHAR(20),
            DealAmount DECIMAL(18,2),
            CloseDate DATETIME,
            Stage NVARCHAR(20),
            Category NVARCHAR(255),
            Quantity INT,
            Profit DECIMAL(18,2)
        )
    """,
    'Activities': """
        CREATE TABLE Activities (
            ActivityID VARCHAR(20) PRIMARY KEY,
            ContactID VARCHAR(20),
            DealID VARCHAR(20),
            ActivityType NVARCHAR(50),
            ActivityDate DATETIME,
            Notes NVARCHAR(500)
        )
    """
}

for table, schema in table_schemas.items():
    cursor.execute(f"IF OBJECT_ID('{table}', 'U') IS NOT NULL DROP TABLE {table};")
    cursor.execute(schema)

print("✅ All tables created successfully!\n")

✅ All tables created successfully!



In [14]:
def get_existing_ids(cursor, table, id_column):
    cursor.execute(f"SELECT {id_column} FROM {table}")
    return set(row[0] for row in cursor.fetchall())

# Get existing ContactIDs from SQL Server
existing_ids = get_existing_ids(cursor, 'Contacts', 'ContactID')

# Filter out duplicates
before_count = len(contacts)
contacts = contacts[~contacts['ContactID'].isin(existing_ids)]
after_count = len(contacts)

print(f"Filtered out {before_count - after_count} existing records from Contacts")

Filtered out 0 existing records from Contacts


In [15]:
# ====================== FIX COLUMN NAMES ======================
# Make sure DataFrame columns match the table exactly
contacts = contacts.rename(columns={'Postal Code': 'PostalCode'})
# Do the same for other tables if needed
leads = leads.rename(columns=lambda x: x.replace(' ', ''))
deals = deals.rename(columns=lambda x: x.replace(' ', ''))
activities = activities.rename(columns=lambda x: x.replace(' ', ''))

# Keep only the columns that exist in the destination tables
contacts = contacts[['ContactID', 'ContactName', 'Segment', 'Country', 'City', 'State', 'PostalCode', 'Region']]
leads = leads[['LeadID', 'ContactID', 'LeadDate', 'LeadSource', 'LeadStatus']]
deals = deals[['DealID', 'ContactID', 'DealAmount', 'CloseDate', 'Stage', 'Category', 'Quantity', 'Profit']]
activities = activities[['ActivityID', 'ContactID', 'DealID', 'ActivityType', 'ActivityDate', 'Notes']]

# ====================== IMPROVED INSERT FUNCTION ======================
def insert_dataframe(cursor, df, table_name):
    if df.empty:
        print(f"No data to insert into {table_name}")
        return
    
    # Use square brackets for safety
    columns = ', '.join([f'[{col}]' for col in df.columns])
    placeholders = ', '.join(['?'] * len(df.columns))
    
    sql = f"INSERT INTO {table_name} ({columns}) VALUES ({placeholders})"
    data = [tuple(row) for row in df.itertuples(index=False, name=None)]
    
    try:
        cursor.executemany(sql, data)
        print(f"✅ Inserted {len(df)} rows into {table_name}")
    except Exception as e:
        print(f"❌ Error inserting into {table_name}: {e}")
        print("Problematic columns:", df.columns.tolist())
        raise

# ====================== INSERT DATA ======================
print("Starting data insertion...\n")

# Remove duplicate primary key rows before inserting
contacts = contacts.drop_duplicates(subset=['ContactID'], keep='first')
leads = leads.drop_duplicates(subset=['LeadID'], keep='first')
deals = deals.drop_duplicates(subset=['DealID'], keep='first')
activities = activities.drop_duplicates(subset=['ActivityID'], keep='first')

# Filter out any existing primary keys to avoid duplicate inserts
existing_contact_ids = get_existing_ids(cursor, 'Contacts', 'ContactID')
existing_lead_ids = get_existing_ids(cursor, 'Leads', 'LeadID')
existing_deal_ids = get_existing_ids(cursor, 'Deals', 'DealID')
existing_activity_ids = get_existing_ids(cursor, 'Activities', 'ActivityID')

contacts = contacts[~contacts['ContactID'].isin(existing_contact_ids)]
leads = leads[~leads['LeadID'].isin(existing_lead_ids)]
deals = deals[~deals['DealID'].isin(existing_deal_ids)]
activities = activities[~activities['ActivityID'].isin(existing_activity_ids)]

insert_dataframe(cursor, contacts, 'Contacts')
insert_dataframe(cursor, leads, 'Leads')
insert_dataframe(cursor, deals, 'Deals')
insert_dataframe(cursor, activities, 'Activities')

conn.commit()
print("\n🎉 ETL PROCESS COMPLETED SUCCESSFULLY!")
print("All data loaded into SQL Server.")

cursor.close()
conn.close()

Starting data insertion...

✅ Inserted 2020 rows into Contacts
✅ Inserted 2020 rows into Leads
✅ Inserted 5009 rows into Deals
✅ Inserted 800 rows into Activities

🎉 ETL PROCESS COMPLETED SUCCESSFULLY!
All data loaded into SQL Server.
